In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

try:
    df = pd.read_csv('superstore_dataset.csv')
    print("Dataset charge avec succes")
except FileNotFoundError:
    print("Fichier non trouve. Creation de donnees simulees...")
    np.random.seed(42)
    n_rows = 5000
    dates = pd.date_range('2019-01-01', '2022-12-31', periods=n_rows)
    categories = ['Furniture', 'Office Supplies', 'Technology']
    sub_categories = ['Chairs', 'Tables', 'Phones', 'Binders', 'Paper', 'Machines']
    states = ['California', 'New York', 'Texas', 'Florida', 'Illinois',
              'Ohio', 'Pennsylvania', 'Georgia', 'North Carolina', 'Michigan']

    df = pd.DataFrame({
        'Order Date': dates,
        'Ship Date': dates + pd.Timedelta(days=np.random.randint(1, 10, n_rows)),
        'Category': np.random.choice(categories, n_rows, p=[0.3, 0.4, 0.3]),
        'Sub-Category': np.random.choice(sub_categories, n_rows),
        'State': np.random.choice(states, n_rows),
        'Sales': np.random.uniform(10, 1000, n_rows),
        'Quantity': np.random.randint(1, 5, n_rows),
        'Discount': np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], n_rows, p=[0.5, 0.2, 0.15, 0.08, 0.05, 0.02]),
        'Profit': np.random.uniform(-50, 200, n_rows),
        'Product Name': [f'Product_{i}' for i in range(n_rows)]
    })
    df['Profit'] = df['Profit'] - df['Discount'] * df['Sales'] * 0.5
    print("Donnees simulees creees")

print(f"Dataset Shape: {df.shape}")
print(f"Colonnes: {df.columns.tolist()}")
df.info()
print(df.describe())
print(df.isnull().sum())

print(f"Doublons: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Nouvelle taille: {df.shape}")

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

critical_cols = ['Sales', 'Profit', 'Category']
df = df.dropna(subset=critical_cols)

date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print(f"Profit Margin moyen : {df['Profit Margin'].mean():.1f}%")
print(f"Annees couvertes    : {sorted(df['Order Year'].unique())}")
display(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head())

monthly_sales  = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()
total_monthly  = df.groupby('Order Month-Year')['Sales'].sum()

def plot_monthly_sales(category='All', show_profit=False):
    plt.figure(figsize=(14, 6))
    if category == 'All':
        if show_profit:
            monthly_profit = df.groupby('Order Month-Year')['Profit'].sum()
            plt.plot(monthly_profit.index.to_timestamp(), monthly_profit.values,
                    marker='s', linewidth=2, markersize=4, color='green', label='Profit')
            plt.ylabel('Profit ($)')
            title = 'Monthly Profit Trend - All Categories'
        else:
            plt.plot(total_monthly.index.to_timestamp(), total_monthly.values,
                    marker='o', linewidth=2, markersize=4, color='steelblue', label='Sales')
            plt.ylabel('Sales ($)')
            title = 'Monthly Sales Trend - All Categories'
    else:
        category_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(category_data['Date'], category_data['Sales'],
                marker='o', linewidth=2, markersize=4, color='coral', label='Sales')
        plt.ylabel('Sales ($)')
        title = f'Monthly Sales Trend - {category}'

    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('Date')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Analyse pour {category}:")
    print(f"  Ventes totales    : ${total_monthly.sum():,.0f}")
    print(f"  Moyenne mensuelle : ${total_monthly.mean():,.0f}")
    print(f"  Mois le plus eleve: {total_monthly.idxmax()} (${total_monthly.max():,.0f})")

categories_list = ['All'] + list(df['Category'].unique())
interact(plot_monthly_sales,
         category=Dropdown(options=categories_list, value='All', description='Category:'),
         show_profit=widgets.Checkbox(value=False, description='Show Profit'))

state_sales  = df.groupby('State')['Sales'].sum().sort_values(ascending=True)
state_profit = df.groupby('State')['Profit'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10, metric='Sales'):
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    if metric == 'Sales':
        top_states = state_sales.tail(top_n)
        color  = 'steelblue'
        title  = f'Top {top_n} States by Sales'
        xlabel = 'Total Sales ($)'
    else:
        top_states = state_profit.tail(top_n)
        color  = 'darkgreen'
        title  = f'Top {top_n} States by Profit'
        xlabel = 'Total Profit ($)'

    plt.barh(range(len(top_states)), top_states.values,
             color=color, alpha=0.8, edgecolor='black')
    plt.yticks(range(len(top_states)), top_states.index)
    plt.xlabel(xlabel, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold')

    for i, (state, value) in enumerate(top_states.items()):
        plt.text(value + max(top_states.values()) * 0.01, i,
                 f'${value:,.0f}', va='center', fontsize=9, fontweight='bold')

    plt.grid(axis='x', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

    total    = top_states.sum()
    all_total = state_sales.sum() if metric == 'Sales' else state_profit.sum()
    print(f"Top {top_n} states total      : ${total:,.0f}")
    print(f"Contribution au total        : {(total/all_total)*100:.1f}%")
    print(f"Etat #1                      : {top_states.index[-1]} (${top_states.iloc[-1]:,.0f})")

interact(plot_top_states,
         top_n=IntSlider(min=5, max=25, value=10, description='Top N:'),
         metric=Dropdown(options=['Sales', 'Profit'], value='Sales', description='Metric:'))

print(f"Top 5 states  : {(state_sales.tail(5).sum() / state_sales.sum())*100:.1f}% des ventes")
print(f"Top 10 states : {(state_sales.tail(10).sum() / state_sales.sum())*100:.1f}% des ventes")

product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)
product_sales  = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

colors1 = plt.cm.viridis(np.linspace(0, 0.8, len(product_profit)))
ax1.barh(range(len(product_profit)), product_profit.values, color=colors1, edgecolor='black')
ax1.set_yticks(range(len(product_profit)))
ax1.set_yticklabels([p[:40] + '...' if len(p) > 40 else p for p in product_profit.index])
ax1.set_xlabel('Total Profit ($)', fontweight='bold')
ax1.set_title('Top 10 Most Profitable Products', fontsize=14, fontweight='bold')
for i, (prod, profit) in enumerate(product_profit.items()):
    ax1.text(profit + max(product_profit.values()) * 0.01, i,
             f'${profit:,.0f}', va='center', fontweight='bold', fontsize=9)

colors2 = plt.cm.plasma(np.linspace(0, 0.8, len(product_sales)))
ax2.barh(range(len(product_sales)), product_sales.values, color=colors2, edgecolor='black')
ax2.set_yticks(range(len(product_sales)))
ax2.set_yticklabels([p[:40] + '...' if len(p) > 40 else p for p in product_sales.index])
ax2.set_xlabel('Total Sales ($)', fontweight='bold')
ax2.set_title('Top 10 Best Selling Products', fontsize=14, fontweight='bold')
for i, (prod, sales) in enumerate(product_sales.items()):
    ax2.text(sales + max(product_sales.values()) * 0.01, i,
             f'${sales:,.0f}', va='center', fontweight='bold', fontsize=9)

for ax in [ax1, ax2]:
    ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.suptitle('Product Performance Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Produit le plus rentable   : {product_profit.index[0][:50]}")
print(f"Profit top 10 produits     : ${product_profit.sum():,.0f}")
print(f"Profit moyen top produits  : ${product_profit.mean():,.0f}")

from numpy.polynomial.polynomial import Polynomial

categories_unique = df['Category'].unique()
colors_cat = {'Furniture': 'coral', 'Office Supplies': 'steelblue', 'Technology': 'seagreen'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for category in categories_unique:
    cat_data = df[df['Category'] == category]
    ax1.scatter(cat_data['Discount'], cat_data['Profit'],
               alpha=0.5, s=30, label=category,
               color=colors_cat.get(category, 'gray'))

x_all    = df['Discount'].values
y_all    = df['Profit'].values
p        = Polynomial.fit(x_all, y_all, 2)
x_smooth = np.linspace(0, 0.8, 100)
ax1.plot(x_smooth, p(x_smooth), 'r--', linewidth=2, label='Tendance', alpha=0.7)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.set_xlabel('Discount Rate', fontweight='bold')
ax1.set_ylabel('Profit ($)', fontweight='bold')
ax1.set_title('Discount vs Profit by Category', fontsize=14, fontweight='bold')
ax1.legend(title='Category')
ax1.grid(True, alpha=0.3, linestyle='--')

discount_bins    = pd.cut(df['Discount'], bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0])
avg_profit_disc  = df.groupby(discount_bins)['Profit'].mean()
colors_bar       = ['green' if x > 0 else 'red' for x in avg_profit_disc.values]
ax2.bar(range(len(avg_profit_disc)), avg_profit_disc.values,
        color=colors_bar, edgecolor='black', alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax2.set_xticks(range(len(avg_profit_disc)))
ax2.set_xticklabels(['0%','1-10%','11-20%','21-30%','31-40%','41-50%'],
                     rotation=45, ha='right')
ax2.set_ylabel('Average Profit ($)', fontweight='bold')
ax2.set_title('Average Profit by Discount Range', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

for threshold in [0.1, 0.2, 0.3, 0.4]:
    high = df[df['Discount'] > threshold]
    print(f"Remise >{threshold*100:.0f}% : {len(high):,} transactions | "
          f"perte: {(high['Profit']<0).mean()*100:.1f}% | "
          f"profit moy: ${high['Profit'].mean():.2f}")

def create_strategic_dashboard():
    fig = plt.figure(figsize=(20, 14))
    gs  = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1 - Tendance mensuelle
    ax1 = fig.add_subplot(gs[0, :2])
    monthly_trend = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_trend.index.to_timestamp(), monthly_trend.values,
            marker='o', linewidth=2, markersize=4, color='#2E86AB')
    ax1.set_title('Monthly Sales Trend', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Sales ($)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3, linestyle='--')

    # 2 - Performance par categorie
    ax2 = fig.add_subplot(gs[0, 2])
    category_perf = df.groupby('Category')[['Sales', 'Profit']].sum()
    x     = range(len(category_perf))
    width = 0.35
    ax2.bar([i - width/2 for i in x], category_perf['Sales'], width,
            label='Sales', color='#A23B72', alpha=0.8)
    ax2.bar([i + width/2 for i in x], category_perf['Profit'], width,
            label='Profit', color='#F18F01', alpha=0.8)
    ax2.set_xticks(x)
    ax2.set_xticklabels(category_perf.index)
    ax2.set_title('Sales & Profit by Category', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

    # 3 - Top etats
    ax3 = fig.add_subplot(gs[1, :2])
    top_states = state_sales.tail(8)
    ax3.barh(range(len(top_states)), top_states.values,
             color='#73AB84', edgecolor='black')
    ax3.set_yticks(range(len(top_states)))
    ax3.set_yticklabels(top_states.index)
    ax3.set_xlabel('Sales ($)')
    ax3.set_title('Top 8 States by Sales', fontsize=14, fontweight='bold')
    for i, (state, value) in enumerate(top_states.items()):
        ax3.text(value + max(top_states.values()) * 0.01, i,
                 f'${value:,.0f}', va='center', fontsize=9)
    ax3.grid(True, alpha=0.3, linestyle='--', axis='x')

    # 4 - Impact remises
    ax4 = fig.add_subplot(gs[1, 2])
    discount_bins = pd.cut(df['Discount'], bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0])
    avg_profit    = df.groupby(discount_bins)['Profit'].mean()
    colors        = ['green' if x > 0 else 'red' for x in avg_profit.values]
    ax4.bar(range(len(avg_profit)), avg_profit.values, color=colors, edgecolor='black')
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax4.set_xticks(range(len(avg_profit)))
    ax4.set_xticklabels(['0%','1-10%','11-20%','21-30%','31-50%'],
                         rotation=45, ha='right', fontsize=8)
    ax4.set_ylabel('Avg Profit ($)')
    ax4.set_title('Average Profit by Discount', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, linestyle='--', axis='y')

    # 5 - Marge par categorie
    ax5 = fig.add_subplot(gs[2, :])
    category_margin = df.groupby('Category')['Profit Margin'].mean().sort_values()
    colors_margin   = ['#D64161' if x < 0 else '#007B7F' for x in category_margin.values]
    ax5.barh(range(len(category_margin)), category_margin.values,
             color=colors_margin, edgecolor='black')
    ax5.set_yticks(range(len(category_margin)))
    ax5.set_yticklabels(category_margin.index)
    ax5.set_xlabel('Average Profit Margin (%)')
    ax5.set_title('Average Profit Margin by Category', fontsize=14, fontweight='bold')
    for i, (cat, margin) in enumerate(category_margin.items()):
        ax5.text(margin + 0.5, i, f'{margin:.1f}%', va='center', fontsize=10)
    ax5.grid(True, alpha=0.3, linestyle='--', axis='x')

    plt.suptitle('US Superstore Strategic Dashboard\nExecutive Summary',
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

create_strategic_dashboard()

total_sales    = df['Sales'].sum()
total_profit   = df['Profit'].sum()
profit_margin  = (total_profit / total_sales) * 100
n_transactions = len(df)
avg_order      = total_sales / n_transactions

print("RAPPORT EXECUTIF")
print(f"Chiffre d'affaires  : ${total_sales:,.0f}")
print(f"Profit total        : ${total_profit:,.0f}")
print(f"Marge beneficiaire  : {profit_margin:.1f}%")
print(f"Transactions        : {n_transactions:,}")
print(f"Panier moyen        : ${avg_order:.2f}")
print(f"Etat top            : {state_sales.index[-1]} (${state_sales.iloc[-1]:,.0f})")
print(f"Categorie leader    : {df.groupby('Category')['Sales'].sum().idxmax()}")
print(f"Remises >20%%        : {(df[df['Discount']>0.2]['Profit']<0).mean()*100:.1f}% en perte")
print("\nRECOMMANDATIONS:")
print("1. Limiter les remises a 20% maximum")
print("2. Renforcer la presence dans les Top 5 etats")
print("3. Capitaliser sur les produits les plus rentables")